# Capillary intrusion benchmark

This notebook builds four straight capillary tubes with different radii, runs the unsteady two-phase solver, and checks whether each tube intrudes at the Young-Laplace pressure.

For now this is the single-angle version of the benchmark. The same setup can later be looped over multiple contact angles.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

cwd = Path.cwd().resolve()
if (cwd / "benchmarks").exists():
    repo_root = cwd
    benchmark_dir = cwd / "benchmarks"
elif cwd.name == "benchmarks":
    benchmark_dir = cwd
    repo_root = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the repo root or from benchmarks/.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(benchmark_dir)

from latteasy._native import build_two_phase_solver, find_two_phase_executable
from latteasy.two_phase import (
    LattEasyUnsteadyRelativePermeability,
    load_density_state,
    read_pressure_steps,
)


In [ ]:
domain_x = 48
domain_y = 72
domain_z = 72

tube_radii = np.array([14.0, 10.0, 7.0, 5.0])
tube_centers = np.array([
    [20, 20],
    [52, 20],
    [20, 52],
    [52, 52],
])
tube_labels = [f"tube {i + 1}" for i in range(len(tube_radii))]

contact_angle_degrees = 156.4
surface_tension = 0.15

rho_f1 = 2.0
rho_f2 = 2.0
rho_d = 0.06
Gc = 0.9

buffer_layers = 4
probe_x = buffer_layers + 8
num_pressure_steps = 12
pressure_span_radius = 4.5
num_cores = os.cpu_count() or 1

convergence = 5e-4
convergence_iter = 250
max_iterations = 3000

theta = np.deg2rad(contact_angle_degrees)
G_ads_f1_s1 = 0.25 * Gc * (rho_f1 - rho_d) * np.cos(theta)

print(f"contact angle = {contact_angle_degrees:.1f} deg")
print(f"G_ads_f1_s1 = {G_ads_f1_s1:.5f}")
print(f"cores = {num_cores}, pressure steps = {num_pressure_steps}, probe_x = {probe_x}")


## Geometry

The array is stored as `(y, z, x)` so the current two-phase wrapper sends the tubes along the flow direction. The pressure ladder is pushed slightly past the smallest nominal radius so the last tube also gets a clean intrusion bracket.

In [ ]:
y = np.arange(domain_y)
z = np.arange(domain_z)
Y, Z = np.meshgrid(y, z, indexing="ij")

geometry = np.zeros((domain_y, domain_z, domain_x), dtype=bool)
tube_masks = []

for center, radius in zip(tube_centers, tube_radii):
    cy, cz = center
    tube_mask = (Y - cy) ** 2 + (Z - cz) ** 2 <= radius ** 2
    geometry |= tube_mask[:, :, None]
    tube_masks.append(tube_mask)

cmap = ListedColormap(["#1f2937", "#f8fafc"])
colors = ["#0f766e", "#2563eb", "#f97316", "#dc2626"]

fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=180, constrained_layout=True)
ax.imshow(geometry[:, :, domain_x // 2].T, origin="lower", cmap=cmap, interpolation="nearest")

for color, label, center, radius in zip(colors, tube_labels, tube_centers, tube_radii):
    cy, cz = center
    ax.add_patch(plt.Circle((cy, cz), radius, fill=False, color=color, lw=2))
    ax.text(cy, cz, f"{label}\nr={int(radius)}", color=color, ha="center", va="center", fontsize=8, weight="bold")

ax.set_title("Four capillary tubes")
ax.set_xlabel("y")
ax.set_ylabel("z")
plt.show()


## Run the simulation

In [ ]:
try:
    two_phase_solver = find_two_phase_executable()
except FileNotFoundError:
    two_phase_solver = build_two_phase_solver()

simulation = LattEasyUnsteadyRelativePermeability(
    geometry,
    cpus=num_cores,
    two_phase_solver_path=two_phase_solver,
    buffer_layers=buffer_layers,
    rho_f1=rho_f1,
    rho_f2=rho_f2,
    rho_d=rho_d,
    minimum_radius=pressure_span_radius,
    num_pressure_steps=num_pressure_steps,
    Gc=Gc,
    G_ads_f1_s1=G_ads_f1_s1,
    convergence=convergence,
    convergence_iter=convergence_iter,
    max_iterations=max_iterations,
    gif_iter=0,
    vtk_iter=0,
    print_geom=False,
    print_stl=False,
)
simulation.run_two_phase()

folder = Path(simulation.folder_path).resolve()
output_dir = folder / "output"

print(f"Simulation folder: {folder}")


## Pressure-step analysis

Each run only tells us that invasion happened between two consecutive pressure steps. That is useful here: for each tube we record the last non-invading step and the first step where the probe plane is at least 50% invaded.

In [ ]:
pressure_steps = read_pressure_steps(output_dir / "output.dat")
rho_files = sorted(output_dir.glob("rho_f1_*.dat"))

intrusion_fraction = np.zeros((len(rho_files), len(tube_radii)))
density_threshold = rho_f1 - 1.0

for run_index, rho_file in enumerate(rho_files):
    density = load_density_state(rho_file, simulation.two_phase_shape)
    probe_slice = density[probe_x]

    for tube_index, tube_mask in enumerate(tube_masks):
        intrusion_fraction[run_index, tube_index] = (probe_slice[tube_mask] >= density_threshold).mean()

young_laplace_pc = 2 * surface_tension * np.abs(np.cos(theta)) / tube_radii
summary_rows = []

for tube_index, (label, radius) in enumerate(zip(tube_labels, tube_radii)):
    invaded = intrusion_fraction[:, tube_index] >= 0.5
    first_full = np.flatnonzero(invaded)

    if len(first_full) == 0:
        lower_pc = pressure_steps[-1]
        upper_pc = np.nan
        midpoint_pc = np.nan
        half_width = np.nan
        theory_in_bracket = False
    else:
        first_run = int(first_full[0])
        lower_index = max(first_run - 1, 0)
        lower_pc = pressure_steps[lower_index]
        upper_pc = pressure_steps[first_run]
        midpoint_pc = 0.5 * (lower_pc + upper_pc)
        half_width = 0.5 * (upper_pc - lower_pc)
        theory_in_bracket = lower_pc <= young_laplace_pc[tube_index] <= upper_pc

    summary_rows.append(
        {
            "tube": label,
            "radius_lu": int(radius),
            "young_laplace_pc_lu": young_laplace_pc[tube_index],
            "lower_pc_lu": lower_pc,
            "upper_pc_lu": upper_pc,
            "midpoint_pc_lu": midpoint_pc,
            "half_width_lu": half_width,
            "theory_in_bracket": theory_in_bracket,
        }
    )

summary = pd.DataFrame(summary_rows)
summary.round(5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), dpi=180, constrained_layout=True)

axes[0].imshow(geometry[:, :, domain_x // 2].T, origin="lower", cmap=cmap, interpolation="nearest")
for color, label, center, radius in zip(colors, tube_labels, tube_centers, tube_radii):
    cy, cz = center
    axes[0].add_patch(plt.Circle((cy, cz), radius, fill=False, color=color, lw=2))
    axes[0].text(cy, cz, f"{label}\nr={int(radius)}", color=color, ha="center", va="center", fontsize=8, weight="bold")
axes[0].set_title("Capillary-tube cross-section")
axes[0].set_xlabel("y")
axes[0].set_ylabel("z")

for color, label, theory_pc, fractions in zip(colors, tube_labels, young_laplace_pc, intrusion_fraction.T):
    axes[1].step(pressure_steps, fractions, where="post", color=color, lw=2, label=label)
    axes[1].axvline(theory_pc, color=color, ls="--", lw=1.5, alpha=0.8)
axes[1].set_title("Invasion at the probe plane")
axes[1].set_xlabel("Capillary pressure [l.u.]")
axes[1].set_ylabel("Invaded fraction")
axes[1].set_ylim(-0.02, 1.02)
axes[1].grid(alpha=0.2)
axes[1].legend(frameon=False, loc="lower right")
plt.show()

fig, ax = plt.subplots(figsize=(5, 5), dpi=180, constrained_layout=True)
ax.plot([0, pressure_steps.max()], [0, pressure_steps.max()], color="#9ca3af", ls="--", lw=1)

for color, row in zip(colors, summary.itertuples(index=False)):
    ax.errorbar(
        row.young_laplace_pc_lu,
        row.midpoint_pc_lu,
        yerr=row.half_width_lu,
        fmt="o",
        ms=8,
        lw=1.8,
        capsize=3,
        color=color,
    )
    ax.text(row.young_laplace_pc_lu, row.midpoint_pc_lu, f"  {row.tube}", color=color, va="center")

ax.set_xlabel("Young-Laplace entry pressure [l.u.]")
ax.set_ylabel("Measured entry pressure [l.u.]")
ax.set_title("Pressure-bracket comparison")
ax.grid(alpha=0.2)
plt.show()
